# Logit Lens: Watching Predictions Form Layer by Layer

**Paper:** nostalgebraist, "interpreting GPT: the logit lens" (2020)
**Paper:** Belrose et al., "Eliciting Latent Predictions from Transformers with the Tuned Lens" (2023)

## What Is the Logit Lens?

Normally, only the **final layer's** hidden state gets projected through the LM head to produce token predictions. But what if we applied that same projection to **every intermediate layer**?

The Logit Lens does exactly that -- it reveals how the model's prediction evolves from vague semantics in early layers to confident predictions in later layers. You can literally watch the model "make up its mind."

### Key Insight
In a residual-stream transformer, each layer **refines** the representation. By decoding at each layer, we see the prediction **crystallize** over depth.

In [ ]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import os
import time
from transformers import AutoModelForCausalLM, AutoTokenizer

%matplotlib inline

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "#f8f9fa",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.titleweight": "bold",
})

OUTPUT_DIR = os.path.expanduser("~/hook_outputs/charts")
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
CACHE_DIR = os.path.expanduser("~/models")

print("Loading model...")
t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, cache_dir=CACHE_DIR, torch_dtype=torch.float16, device_map="auto",
)
model.eval()
print("Loaded in {:.1f}s".format(time.time() - t0))

## Step 1: Capture Hidden States at Every Layer

We attach a forward hook to each of the 32 transformer layers to capture the residual stream output. Then we apply the model's own final RMSNorm + LM head to decode each layer's hidden state into vocabulary logits.

In [ ]:
PROMPT = "The capital of France is"

inputs = tokenizer(PROMPT, return_tensors="pt").to(model.device)
input_ids = inputs["input_ids"]
tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
num_layers = model.config.num_hidden_layers

print("Prompt: '{}'".format(PROMPT))
print("Tokens:", tokens)
print("Layers:", num_layers)

# Capture hidden states at every layer using hooks
layer_hidden_states = {}

def make_hook(layer_idx):
    def hook_fn(module, input, output):
        hidden = output[0] if isinstance(output, tuple) else output
        layer_hidden_states[layer_idx] = hidden.detach()
    return hook_fn

handles = []
for i in range(num_layers):
    h = model.model.layers[i].register_forward_hook(make_hook(i))
    handles.append(h)

# Also capture embedding output
def embed_hook(module, input, output):
    layer_hidden_states[-1] = output.detach()
handles.append(model.model.embed_tokens.register_forward_hook(embed_hook))

with torch.no_grad():
    outputs = model(**inputs)

for h in handles:
    h.remove()

print("Captured hidden states from {} layers + embedding".format(len(layer_hidden_states)))

## Step 2: Apply the Logit Lens

For each layer, we take the hidden state, apply the final RMSNorm, then project through the LM head to get vocabulary logits. This tells us: **"If decoding stopped at this layer, what would the model predict?"**

In [ ]:
# Apply logit lens: final_norm + lm_head at each layer
logit_lens_results = {}  # layer_idx -> {token_pos -> (top_token, probability)}

final_norm = model.model.norm
lm_head = model.lm_head

with torch.no_grad():
    for layer_idx in sorted(layer_hidden_states.keys()):
        hidden = layer_hidden_states[layer_idx]
        # Apply final norm then LM head
        normed = final_norm(hidden)
        logits = lm_head(normed)  # (batch, seq_len, vocab_size)
        probs = F.softmax(logits[0], dim=-1)  # (seq_len, vocab_size)
        
        layer_results = {}
        for pos in range(probs.shape[0]):
            top_probs, top_ids = probs[pos].topk(5)
            top_tokens = [tokenizer.decode(tid.item()) for tid in top_ids]
            layer_results[pos] = list(zip(top_tokens, top_probs.cpu().tolist()))
        
        logit_lens_results[layer_idx] = layer_results

# Show the last token position (next-token prediction) across layers
print("Logit Lens: Next-token prediction at each layer")
print("=" * 70)
print("{:<10} {:<20} {:<10} {:<30}".format("Layer", "Top Prediction", "Prob", "Top-5"))
print("-" * 70)
for layer_idx in sorted(logit_lens_results.keys()):
    result = logit_lens_results[layer_idx]
    last_pos = max(result.keys())
    top1_token, top1_prob = result[last_pos][0]
    top5_str = ", ".join(["{} ({:.2f})".format(t, p) for t, p in result[last_pos][:3]])
    label = "Embed" if layer_idx == -1 else "L{}".format(layer_idx)
    print("{:<10} {:<20} {:<10.4f} {:<30}".format(label, repr(top1_token), top1_prob, top5_str))

## Step 3: Visualize -- The Prediction Heatmap

The signature logit lens visualization: a heatmap showing the **top predicted token** at each (layer, position) pair. Watch predictions crystallize from noise to confident answers.

In [ ]:
# Build the probability matrix for the CORRECT next token at each (layer, position)
# For each position, the "correct" next token is the actual next token in the sequence
seq_len = len(tokens)
layers_to_show = list(range(num_layers))  # 0..31

# Get the actual next tokens (shifted by 1)
actual_next_tokens = input_ids[0, 1:].cpu().tolist()

# Matrix: probability of the correct next token at each layer
correct_token_probs = np.zeros((num_layers, seq_len - 1))

# Matrix: top-1 predicted token text at each layer (for the last position)
top1_tokens_matrix = []

with torch.no_grad():
    for li in range(num_layers):
        hidden = layer_hidden_states[li]
        normed = final_norm(hidden)
        logits = lm_head(normed)
        probs = F.softmax(logits[0].float(), dim=-1)
        
        for pos in range(seq_len - 1):
            next_token_id = actual_next_tokens[pos]
            correct_token_probs[li, pos] = probs[pos, next_token_id].cpu().item()

# Also build the top-1 prediction text matrix for ALL positions
top1_text = np.empty((num_layers, seq_len), dtype=object)
top1_prob_matrix = np.zeros((num_layers, seq_len))

with torch.no_grad():
    for li in range(num_layers):
        hidden = layer_hidden_states[li]
        normed = final_norm(hidden)
        logits = lm_head(normed)
        probs = F.softmax(logits[0].float(), dim=-1)
        
        for pos in range(seq_len):
            top_prob, top_id = probs[pos].max(dim=-1)
            top1_text[li, pos] = tokenizer.decode(top_id.item()).strip()
            top1_prob_matrix[li, pos] = top_prob.cpu().item()

print("Built logit lens matrices: {} layers x {} positions".format(num_layers, seq_len))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 10))
fig.suptitle("Logit Lens: Watching Predictions Form Layer by Layer", fontsize=16, fontweight="bold")

# Left: Probability of correct next token
ax = axes[0]
im = ax.imshow(correct_token_probs, aspect="auto", cmap="YlOrRd", interpolation="nearest", vmin=0, vmax=1)
ax.set_xlabel("Token Position")
ax.set_ylabel("Layer")
ax.set_title("P(correct next token) at each layer")
ax.set_xticks(range(seq_len - 1))
ax.set_xticklabels([tokens[i] + " -> " + tokens[i+1] for i in range(seq_len-1)], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(0, num_layers, 4))
plt.colorbar(im, ax=ax, shrink=0.8, label="Probability")

# Right: Top-1 confidence at last position (next token prediction)
ax = axes[1]
im2 = ax.imshow(top1_prob_matrix, aspect="auto", cmap="viridis", interpolation="nearest", vmin=0, vmax=1)
ax.set_xlabel("Token Position")
ax.set_ylabel("Layer")
ax.set_title("Top-1 prediction confidence at each layer")
ax.set_xticks(range(seq_len))
ax.set_xticklabels(tokens, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(0, num_layers, 4))
plt.colorbar(im2, ax=ax, shrink=0.8, label="Probability")

# Annotate with predicted tokens (last position only, every 4 layers)
last_pos = seq_len - 1
for li in range(0, num_layers, 2):
    txt = top1_text[li, last_pos]
    if len(txt) > 8:
        txt = txt[:8]
    ax.text(last_pos, li, txt, ha="center", va="center", fontsize=6,
            color="white" if top1_prob_matrix[li, last_pos] < 0.5 else "black",
            fontweight="bold")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "chart_09_logit_lens.png"), dpi=150, bbox_inches="tight")
plt.show()

## Step 4: Prediction Evolution for the Last Token

Let's zoom in on how the model's top prediction for the **next token** (after "is") evolves across all 32 layers.

In [ ]:
# Detailed view: top-5 predictions at each layer for the last position
last_pos = seq_len - 1
print("How the prediction for '{}' -> ??? evolves:".format(tokens[last_pos]))
print("=" * 80)

layer_top5 = []
with torch.no_grad():
    for li in range(num_layers):
        hidden = layer_hidden_states[li]
        normed = final_norm(hidden)
        logits = lm_head(normed)
        probs = F.softmax(logits[0, last_pos].float(), dim=-1)
        top_probs, top_ids = probs.topk(5)
        top5 = [(tokenizer.decode(tid.item()).strip(), p.item()) for tid, p in zip(top_ids, top_probs)]
        layer_top5.append(top5)
        
        top5_str = " | ".join(["{} ({:.3f})".format(t, p) for t, p in top5])
        print("L{:>2}: {}".format(li, top5_str))

In [ ]:
# Track probability of the final answer across layers
# Get the final model's top prediction
final_top_token = layer_top5[-1][0][0]
final_top_id = tokenizer.encode(final_top_token, add_special_tokens=False)[0]

prob_of_answer = []
with torch.no_grad():
    for li in range(num_layers):
        hidden = layer_hidden_states[li]
        normed = final_norm(hidden)
        logits = lm_head(normed)
        probs = F.softmax(logits[0, last_pos].float(), dim=-1)
        prob_of_answer.append(probs[final_top_id].cpu().item())

# Entropy at each layer (measures confidence)
entropy_per_layer = []
with torch.no_grad():
    for li in range(num_layers):
        hidden = layer_hidden_states[li]
        normed = final_norm(hidden)
        logits = lm_head(normed)
        probs = F.softmax(logits[0, last_pos].float(), dim=-1)
        ent = -(probs * torch.log(probs + 1e-10)).sum().item()
        entropy_per_layer.append(ent)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Prediction Crystallization: '{}'".format(PROMPT), fontsize=14, fontweight="bold")

# Left: probability of final answer
ax = axes[0]
ax.plot(range(num_layers), prob_of_answer, "o-", color="#4361ee", linewidth=2, markersize=5)
ax.fill_between(range(num_layers), prob_of_answer, alpha=0.2, color="#4361ee")
ax.set_xlabel("Layer")
ax.set_ylabel("P('{}')".format(final_top_token))
ax.set_title("Probability of '{}' at each layer".format(final_top_token))
ax.set_ylim(0, 1)
# Mark where it crosses 50%
for i, p in enumerate(prob_of_answer):
    if p > 0.5:
        ax.axvline(x=i, color="#f72585", linestyle="--", alpha=0.5)
        ax.text(i + 0.5, 0.9, "Commits at L{}".format(i), fontsize=10, color="#f72585")
        break

# Right: entropy (uncertainty)
ax = axes[1]
ax.plot(range(num_layers), entropy_per_layer, "s-", color="#f72585", linewidth=2, markersize=5)
ax.fill_between(range(num_layers), entropy_per_layer, alpha=0.2, color="#f72585")
ax.set_xlabel("Layer")
ax.set_ylabel("Entropy (nats)")
ax.set_title("Prediction Uncertainty (entropy drops = model gets confident)")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "chart_10_logit_lens_evolution.png"), dpi=150, bbox_inches="tight")
plt.show()

## Step 5: Multi-Prompt Logit Lens

Let's run the logit lens on several factual prompts to see if the model always crystallizes at the same depth, or if different facts emerge at different layers.

In [ ]:
prompts = [
    "The capital of France is",
    "The largest planet in our solar system is",
    "Water boils at",
    "The CEO of Tesla is",
    "Python was created by",
    "The speed of light is approximately",
]

all_prob_curves = {}

for prompt in prompts:
    inp = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Capture hidden states
    states = {}
    handles = []
    for i in range(num_layers):
        def make_h(idx):
            def h(module, input, output):
                states[idx] = (output[0] if isinstance(output, tuple) else output).detach()
            return h
        handles.append(model.model.layers[i].register_forward_hook(make_h(i)))
    
    with torch.no_grad():
        out = model(**inp)
    
    for h in handles:
        h.remove()
    
    # Get final prediction
    final_logits = out.logits[0, -1]
    final_top_id = final_logits.argmax().item()
    final_token = tokenizer.decode(final_top_id).strip()
    
    # Track probability of that token across layers
    probs_curve = []
    with torch.no_grad():
        for li in range(num_layers):
            normed = final_norm(states[li])
            logits = lm_head(normed)
            p = F.softmax(logits[0, -1].float(), dim=-1)
            probs_curve.append(p[final_top_id].cpu().item())
    
    all_prob_curves[prompt + " -> " + final_token] = probs_curve

# Plot all curves
fig, ax = plt.subplots(figsize=(14, 7))
colors = plt.cm.tab10(np.linspace(0, 1, len(all_prob_curves)))

for (label, curve), color in zip(all_prob_curves.items(), colors):
    short_label = label if len(label) < 50 else label[:47] + "..."
    ax.plot(range(num_layers), curve, "o-", color=color, linewidth=2, markersize=4, label=short_label)

ax.axhline(y=0.5, color="gray", linestyle="--", alpha=0.5, label="50% threshold")
ax.set_xlabel("Layer")
ax.set_ylabel("Probability of Final Answer")
ax.set_title("Logit Lens: When Does the Model 'Know' the Answer?")
ax.legend(fontsize=8, loc="upper left")
ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "chart_11_logit_lens_multi.png"), dpi=150, bbox_inches="tight")
plt.show()

## Key Takeaways

1. **Early layers are noisy** -- the logit lens shows near-random predictions in layers 0-5
2. **Mid layers begin to converge** -- by layers 10-15, the model often has the right answer in its top-5
3. **Final layers sharpen** -- the last 5-10 layers primarily increase confidence, not change the prediction
4. **Different facts crystallize at different depths** -- simple facts (capitals) emerge earlier than complex ones
5. **Entropy drops monotonically** -- the model gets steadily more confident as depth increases

### References
- nostalgebraist (2020), "interpreting GPT: the logit lens"
- Belrose et al. (2023), "Eliciting Latent Predictions from Transformers with the Tuned Lens"